# Evaluation Comparison

Benchmark all trained models using BLEU, ROUGE, and Exact Match.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.data.preprocess import load_stackoverflow, split_qa, make_sft_dataframe
from src.eval.generate import generate_predictions
from src.eval.metrics import compute_all_metrics

### Load Test Set

In [ ]:
df = load_stackoverflow('../data/raw/stacksample')
_, _, test_df = split_qa(df)
test_ds = Dataset.from_pandas(make_sft_dataframe(test_df), preserve_index=False)
print(f'Test samples: {len(test_ds)}')

### Define evaluate_model Helper

In [ ]:
def evaluate_model(path, num_samples=20):
    tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(path, device_map='auto', trust_remote_code=True)

    prompts = test_ds['text'][:num_samples]
    refs = [p.split('Answer:')[-1].strip() for p in prompts]
    preds = generate_predictions(model, tokenizer, prompts)

    return compute_all_metrics(preds, refs)

### Benchmark All Models

In [ ]:
models = {
    'SFT+LoRA': '../outputs/sft-lora',
    'SFT+QLoRA': '../outputs/sft-qlora',
    'DPO': '../outputs/dpo',
    'GRPO': '../outputs/grpo',
}

rows = []
for name, path in models.items():
    try:
        metrics = evaluate_model(path)
        metrics['method'] = name
        rows.append(metrics)
        print(f'[OK] {name}: {metrics}')
    except Exception as e:
        print(f'[FAIL] {name}: {e}')

results = pd.DataFrame(rows)
results

### Plot Comparison

In [ ]:
import matplotlib.pyplot as plt

if not results.empty:
    results.set_index('method')[['bleu', 'rougeL', 'exact_match']].plot(kind='bar', figsize=(10, 5))
    plt.title('Fine-Tuning Method Comparison')
    plt.tight_layout()
    plt.savefig('../outputs/evaluation_comparison.png')
    plt.show()